# Update meta data and objects 

In [1]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat), 
        
        # Data 
        library(tidyverse), 
        library(data.table), 
        
        # Plotting 
        library(patchwork), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
    
)

Warning message:
“package ‘ggplot2’ was built under R version 4.1.3”
Warning message:
“package ‘tibble’ was built under R version 4.1.3”
Warning message:
“package ‘tidyr’ was built under R version 4.1.3”
Warning message:
“package ‘readr’ was built under R version 4.1.3”
Warning message:
“package ‘purrr’ was built under R version 4.1.3”
Warning message:
“package ‘dplyr’ was built under R version 4.1.3”
Warning message:
“package ‘stringr’ was built under R version 4.1.3”
Warning message:
“package ‘forcats’ was built under R version 4.1.3”
Warning message:
“package ‘lubridate’ was built under R version 4.1.3”
Warning message:
“package ‘data.table’ was built under R version 4.1.3”
Warning message:
“package ‘patchwork’ was built under R version 4.1.3”


In [2]:
# Configure reticulate 
use_condaenv(condaenv='p.3.8.12-FD20220623HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

python:         /nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT/bin/python
libpython:      /nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT/lib/libpython3.8.so
pythonhome:     /nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT:/nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT
version:        3.8.12 | packaged by conda-forge | (default, Jan 30 2022, 23:42:07)  [GCC 9.4.0]
numpy:          /nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT/lib/python3.8/site-packages/numpy
numpy_version:  1.22.4

NOTE: Python version was forced by use_python() function

In [3]:
options(warn=-1)

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

In [6]:
# Source files 
source("bin/seurat_qc.R")

In [7]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Files and parameters

In [8]:
# Seurat object 
pp_rds_file <- "data/object/pp.rds"
pp_subset_rds_file <- "data/object/pp_subset.rds"

# Meta data 
cell_type_leiden_subset_file_1 <- "result/annotation/sbusets/lymphocyte_r.csv"
cell_type_leiden_subset_file_2 <- "result/annotation/sbusets/myeloid_r.csv"
cell_type_leiden_subset_file_3 <- "result/annotation/sbusets/structural_r.csv"

# UMAP 
umap_file <- "data/object/subsets/scvi/components/umap.csv"

# Import 

In [9]:
so <- readRDS(pp_rds_file)

In [10]:
cell_type_leiden_subset_1 <- read.csv(cell_type_leiden_subset_file_1, row.names=1) %>% dplyr::mutate(domain_leiden_subset='lymphocyte')
cell_type_leiden_subset_2 <- read.csv(cell_type_leiden_subset_file_2, row.names=1) %>% dplyr::mutate(domain_leiden_subset='myeloid')
cell_type_leiden_subset_3 <- read.csv(cell_type_leiden_subset_file_3, row.names=1) %>% dplyr::mutate(domain_leiden_subset='structural')

In [11]:
cell_type_leiden_subset <- rbind(cell_type_leiden_subset_1, cell_type_leiden_subset_2, cell_type_leiden_subset_3)

In [12]:
umap <- read.csv(umap_file, row.names=1)
colnames(umap) <- c("UMAP_1", "UMAP_2")

# Meta data 

In [ ]:
so <- AddMetaData(so, cell_type_leiden_subset)

In [ ]:
so <- set_factor_levels(so)

# Subset 

In [ ]:
so <- subset(so, subset=cell_id %in% rownames(umap))

# UMAP

In [ ]:
# Add meta data
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

# UMAP 

In [ ]:
dplot_1 <- dplot(so, reduction="umap", group_by="cell_type_leiden", alpha=1, pt_size=1.5, label=TRUE, label_size=16, label_box=TRUE) + ggtitle("Cell type leiden") + scale_color_manual(values=color$cell_type_leiden, na.value="grey50") + scale_fill_manual(values=color$cell_type_leiden, na.value="grey50") + theme(legend.position="none")
dplot_2 <- dplot(so, reduction="umap", group_by="cell_type_leiden_subset", alpha=1, pt_size=1.5, label=TRUE, label_size=16, label_box=TRUE, label_box_parse=TRUE) + ggtitle("Cell type leiden") + scale_color_manual(values=color$cell_type_leiden_subset_r, na.value="grey50") + scale_fill_manual(values=color$cell_type_leiden_subset_r, na.value="grey50") + theme(legend.position="none")
dplot_3 <- dplot(so, reduction="umap", group_by="patient_id", alpha=1, pt_size=1.5) + ggtitle("Patient ID") + scale_color_manual(values=color$patient_id, na.value="grey50")
dplot_4 <- dplot(so, reduction="umap", group_by="hto_demux_class", alpha=1, pt_size=1.5) + ggtitle("Tissue") + scale_color_manual(values=color$hto_demux_class, na.value="grey50")
dplot_5 <- dplot(so, reduction="umap", group_by="time_point", alpha=1, pt_size=1.5) + ggtitle("Time point") + scale_color_manual(values=color$time_point, na.value="grey50")
dplot_6 <- dplot(so, reduction="umap", group_by="cc_phase_class", alpha=1, pt_size=1.5) + ggtitle("Cell cycle") + scale_color_manual(values=color$cc_phase_class, na.value="grey50")
dplot_7 <- dplot(so, reduction="umap", group_by="digestion_protocol", alpha=1, pt_size=1.5) + ggtitle("Digestion protocol") + scale_color_manual(values=color$digestion_protocol, na.value="grey50")
dplot_8 <- dplot(subset(so, subset=genotype_class %in% c("Host", "Donor", "unassigned", "doublet")), reduction="umap", group_by="genotype_class", alpha=1, pt_size=1.5) + ggtitle("Genotype") + scale_color_manual(values=color$genotype_class, na.value="grey50")
fplot_1 <- fplot(so, reduction="umap", features="cc_aa_alignment_size", alpha=1, pt_size=1.5) + ggtitle("TCR expansion") 

In [ ]:
options(repr.plot.width=4*10, repr.plot.height=1*10)

dplot_1 + dplot_2 + dplot_3 + dplot_4 + plot_layout(ncol=4) 

In [ ]:
options(repr.plot.width=4*10, repr.plot.height=2*10)

dplot_5 + dplot_6 + dplot_7 + dplot_8 + fplot_1 + plot_layout(ncol=4)

In [ ]:
options(repr.plot.width=30, repr.plot.height=10)

dplot_8 <- dplot(so, reduction="umap", group_by="celltypist_immun_low", alpha=1, pt_size=1.5) + ggtitle("CellTypist Immun labels (low)")
dplot_8

In [ ]:
options(repr.plot.width=30, repr.plot.height=10)

dplot_9 <- dplot(so, reduction="umap", group_by="celltypist_immun_high", alpha=1, pt_size=1.5) + ggtitle("CellTypist Immun labels (high)")
dplot_9

# Save result 

In [ ]:
cnt <- GetAssayData(so, assay="RNA", slot="counts")
so <- CreateSeuratObject(counts=cnt, meta.data=so@meta.data, project="FD20220623HSCT")
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

In [ ]:
saveRDS(so, pp_subset_rds_file)